# Laboratory 04 — The microscopic origin of pressure

In this laboratory you will watch a steady macroscopic pressure assemble itself out of
discrete, irregular molecular impacts — and measure exactly how steady it is.

Work through it in order. Where the notebook asks you to predict, write your prediction in
the cell provided **before** running the next cell. That is not a ritual: a prediction you
have committed to is the only reliable way to discover that you were wrong.

## Model specification

| | |
|---|---|
| **System** | $N$ point particles of mass $m$ in a rectangular 2D container |
| **Dynamics** | free flight, perfectly elastic wall collisions, no particle–particle interaction |
| **Boundary** | rigid fixed walls; a piston is modelled by changing the box size |
| **Ensemble** | approximately microcanonical — energy exactly conserved, velocities drawn from Maxwell–Boltzmann at $T$ |
| **Ignored** | intermolecular forces, particle size, quantum effects, gravity |
| **Valid when** | dilute classical regime |
| **Failure modes** | high density, low temperature, strong interactions |

All the physics lives in `thermolab.kinetics` — open it and read it. Nothing in this course
is hidden inside a framework.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from thermolab import kinetics
from thermolab.constants import K_B
from thermolab.validation import scaling_exponent, seed_study

ARGON_MASS = 39.948 * 1.66053906660e-27  # kg
BOX = (1e-6, 1e-6)  # metres; area = 1e-12 m^2
AREA = float(np.prod(BOX))

# Every stochastic function takes its generator explicitly, so results are reproducible
# and no hidden global state can leak between cells.
rng = np.random.default_rng(2024)

print(f"k_B = {K_B:.6e} J/K")
print(f"box area = {AREA:.2e} m^2")

## Part 1 — Watch the impacts arrive

Start small enough to see individual events. Twenty particles, room temperature.

In [ ]:
state = kinetics.initialise_gas(20, BOX, temperature=300.0, mass=ARGON_MASS, rng=rng)
dt = kinetics.max_stable_dt(state)
result = kinetics.simulate(state, dt=dt, n_steps=3000)

expected = kinetics.ideal_gas_pressure(20, 300.0, AREA)
running = np.cumsum(result.impulses) / (result.times * result.wall_measure)

fig, (top, bottom) = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
top.plot(result.times * 1e9, result.impulses, lw=0.6)
top.set_ylabel("impulse per step\n(kg m/s)")
top.set_title("N = 20: individual impacts (top) and their running average (bottom)")
bottom.plot(result.times * 1e9, running, lw=1.2, label="running average")
bottom.axhline(expected, color="crimson", ls="--", label=r"$N k_B T / V$")
bottom.set_xlabel("time (ns)")
bottom.set_ylabel("pressure (N/m)")
bottom.legend()
plt.tight_layout()
plt.show()

print(f"measured  {result.pressure():.4e} N/m")
print(f"predicted {expected:.4e} N/m")

The top panel is chaos: nothing about it looks like a law of physics. The bottom panel is
the same data, accumulated. Notice that the running average is still visibly wandering even
after three thousand steps — with twenty particles, "pressure" is a shaky idea.

### Predict

Before running the next cell, write down what the bottom panel will look like with $N = 2000$
instead of $20$. Be specific: will it approach the red line faster, or just wobble less?

**Your prediction:**

*(write here before running the next cell)*

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6), sharey=False)

for ax, n_particles in zip(axes, (20, 200, 2000), strict=True):
    state = kinetics.initialise_gas(n_particles, BOX, 300.0, ARGON_MASS, rng)
    run = kinetics.simulate(state, dt=kinetics.max_stable_dt(state), n_steps=3000)
    predicted = kinetics.ideal_gas_pressure(n_particles, 300.0, AREA)
    trace = np.cumsum(run.impulses) / (run.times * run.wall_measure)

    ax.plot(run.times * 1e9, trace / predicted, lw=1.0)
    ax.axhline(1.0, color="crimson", ls="--")
    ax.set_ylim(0.5, 1.5)
    ax.set_title(f"N = {n_particles}")
    ax.set_xlabel("time (ns)")

axes[0].set_ylabel(r"measured $P$ / predicted $P$")
plt.tight_layout()
plt.show()

Each panel is plotted on the same vertical scale, as a *fraction* of the predicted pressure.
The convergence gets dramatically tighter with $N$ — and that tightening is the phenomenon we
are going to quantify in Part 3.

## Part 2 — Is the pressure really $N k_B T / V$?

Measure, do not assume. Vary one control at a time and compare with the equation of state.

In [ ]:
def measure_pressure(n_particles, temperature, box, mass, rng, n_steps=4000):
    """Time-averaged pressure from wall impulses, in the same units as the ideal gas law."""
    state = kinetics.initialise_gas(n_particles, box, temperature, mass, rng)
    run = kinetics.simulate(state, dt=kinetics.max_stable_dt(state), n_steps=n_steps)
    return run.pressure()


sizes = np.array([50, 100, 200, 400, 800])
measured = np.array([measure_pressure(int(n), 300.0, BOX, ARGON_MASS, rng) for n in sizes])
predicted = np.array([kinetics.ideal_gas_pressure(int(n), 300.0, AREA) for n in sizes])

for n, m, p in zip(sizes, measured, predicted, strict=True):
    print(f"N = {n:4d}   measured {m:.4e}   predicted {p:.4e}   ratio {m / p:.4f}")

In [ ]:
temperatures = np.array([100.0, 200.0, 300.0, 400.0, 600.0])
p_of_t = np.array([measure_pressure(300, float(t), BOX, ARGON_MASS, rng) for t in temperatures])

# Doubling the particle mass at fixed temperature: the question that catches most people.
p_light = measure_pressure(300, 300.0, BOX, ARGON_MASS, rng)
p_heavy = measure_pressure(300, 300.0, BOX, 2 * ARGON_MASS, rng)

fig, (left, right) = plt.subplots(1, 2, figsize=(11, 4))
left.plot(sizes, measured, "o", label="measured")
left.plot(sizes, predicted, "-", label=r"$N k_B T / V$")
left.set_xlabel("N"), left.set_ylabel("pressure (N/m)"), left.legend()
left.set_title("pressure is proportional to N")

right.plot(temperatures, p_of_t, "o", label="measured")
right.plot(temperatures, 300 * K_B * temperatures / AREA, "-", label=r"$N k_B T / V$")
right.set_xlabel("T (K)"), right.set_ylabel("pressure (N/m)"), right.legend()
right.set_title("and proportional to T")
plt.tight_layout()
plt.show()

print(f"mass  m : P = {p_light:.4e} N/m")
print(f"mass 2m : P = {p_heavy:.4e} N/m      ratio = {p_heavy / p_light:.4f}")

The mass ratio comes out at 1, not 2. Heavier particles hit harder — but at fixed temperature
they move more slowly by exactly the compensating factor, since
$\langle v_x^2\rangle = k_B T/m$. In $P = N m \langle v_x^2\rangle / V$ the mass cancels
identically. If your prediction was that heavier gas pushes harder, this is the number to sit
with for a moment.

## Part 3 — How steady is steady? The $N^{-1/2}$ law

Now we quantify the opening puzzle. There is a subtlety worth understanding before measuring.

Our particles never collide with each other, so each one bounces periodically and delivers a
perfectly regular train of impulses. The long-time average pressure is therefore fixed by the
microstate alone, and the wobble you see *within* one run is largely an artefact of where the
averaging window happens to cut a bounce.

The physically meaningful fluctuation is across **independently drawn microstates** — which
is what a gas in contact with a thermal reservoir actually explores. For that we must stop
rescaling each sample to exactly $T$ (`fix_temperature=False`), because that rescaling
removes the very energy fluctuation we want to measure.

In [ ]:
def sampled_pressure(n_particles, rng, n_steps=3000):
    """Pressure of one independently drawn microstate — energy is NOT rescaled."""
    state = kinetics.initialise_gas(n_particles, BOX, 300.0, ARGON_MASS, rng, fix_temperature=False)
    return kinetics.simulate(state, dt=kinetics.max_stable_dt(state), n_steps=n_steps).pressure()


def relative_spread(n_particles, n_samples=40, base_seed=7):
    seeds = np.random.SeedSequence(base_seed).spawn(n_samples)
    values = np.array([sampled_pressure(n_particles, np.random.default_rng(s)) for s in seeds])
    return float(values.std(ddof=1) / values.mean())


fluct_sizes = np.array([25, 50, 100, 200, 400])
spreads = np.array([relative_spread(int(n)) for n in fluct_sizes])
exponent = scaling_exponent(fluct_sizes, spreads)

for n, s in zip(fluct_sizes, spreads, strict=True):
    print(f"N = {n:4d}   sigma_P/<P> = {s:.4f}   1/sqrt(N) = {1 / np.sqrt(n):.4f}")
print(f"\nfitted exponent = {exponent:.3f}   (theory: -0.5)")

In [ ]:
plt.figure(figsize=(6, 4.5))
plt.loglog(fluct_sizes, spreads, "o", label="measured")
plt.loglog(fluct_sizes, 1 / np.sqrt(fluct_sizes), "-", label=r"$N^{-1/2}$")
plt.xlabel("N")
plt.ylabel(r"$\sigma_P / \langle P \rangle$")
plt.title(f"relative pressure fluctuation (fitted slope {exponent:.2f})")
plt.legend()
plt.tight_layout()
plt.show()

for n in (1e2, 1e6, 6.022e23):
    print(f"N = {n:.3e}  ->  relative fluctuation {1 / np.sqrt(n):.3e}")

There is the answer to the puzzle. The collisions never became gentle — a mole of gas simply
averages over so many of them that the fractional jitter is $4 \times 10^{-13}$, far below
the noise floor of any instrument.

The reason the exponent is exactly $-1/2$: the time-averaged pressure is proportional to the
sum of $2N$ squared velocity components, which is chi-square distributed with $2N$ degrees of
freedom, whose relative standard deviation is $\sqrt{2/(2N)} = N^{-1/2}$.

## Part 4 — Automated checks

A simulation you have not checked is a picture, not evidence. These are the same assertions
that run in the project's test suite.

In [ ]:
check_state = kinetics.initialise_gas(400, BOX, 300.0, ARGON_MASS, np.random.default_rng(1))
check_run = kinetics.simulate(check_state, dt=kinetics.max_stable_dt(check_state), n_steps=4000)

# 1. Energy conservation — elastic reflection only flips signs, so this must be exact.
energy_drift = np.max(np.abs(check_run.kinetic_energy - check_state.kinetic_energy))
assert energy_drift / check_state.kinetic_energy < 1e-12

# 2. Particle number and containment.
assert check_run.final_state.n_particles == 400
assert np.all(check_run.final_state.positions >= 0)
assert np.all(check_run.final_state.positions <= check_run.final_state.box)

# 3. Equipartition: <E> = (d/2) k_B T, so k_B T in two dimensions.
mean_energy = check_state.kinetic_energy / check_state.n_particles
assert abs(mean_energy / kinetics.mean_kinetic_energy(300.0, 2) - 1) < 1e-9

# 4. The equation of state, across independent seeds and with an honest error bar.
study = seed_study(
    lambda r: measure_pressure(400, 300.0, BOX, ARGON_MASS, r, n_steps=4000), n_seeds=8
)
target = kinetics.ideal_gas_pressure(400, 300.0, AREA)
assert study.agrees_with(target, n_sigma=3.0)

print(f"energy drift            {energy_drift / check_state.kinetic_energy:.2e}  (must be ~0)")
print(f"mean energy / (d/2)k_BT {mean_energy / kinetics.mean_kinetic_energy(300.0, 2):.9f}")
print(f"pressure  {study.mean:.4e} +/- {study.standard_error:.1e}  vs  {target:.4e}")
print("\nall checks passed")

## Part 5 — Explore it yourself

The sliders below let you vary the controls freely. Two experiments worth doing:

1. Shrink the box at fixed $N$ and $T$ and watch the collision rate — this is a piston
   compressing a gas, and it is where module 5 begins.
2. Find the smallest $N$ for which you would still be willing to call the pressure "steady",
   and say what standard you used to decide.

In [ ]:
import ipywidgets as widgets


def explore(n_particles=200, temperature=300.0, mass_amu=39.948, box_microns=1.0):
    box = (box_microns * 1e-6, box_microns * 1e-6)
    area = float(np.prod(box))
    mass = mass_amu * 1.66053906660e-27
    state = kinetics.initialise_gas(n_particles, box, temperature, mass, np.random.default_rng(0))
    run = kinetics.simulate(state, dt=kinetics.max_stable_dt(state), n_steps=2500)
    trace = np.cumsum(run.impulses) / (run.times * run.wall_measure)
    target = kinetics.ideal_gas_pressure(n_particles, temperature, area)

    fig, (left, right) = plt.subplots(1, 2, figsize=(11, 3.6))
    left.scatter(state.positions[:, 0] * 1e6, state.positions[:, 1] * 1e6, s=6)
    left.set_xlim(0, box_microns), left.set_ylim(0, box_microns)
    left.set_xlabel("x (um)"), left.set_ylabel("y (um)"), left.set_title("initial positions")
    right.plot(run.times * 1e9, trace, lw=1.0)
    right.axhline(target, color="crimson", ls="--")
    right.set_xlabel("time (ns)"), right.set_ylabel("pressure (N/m)")
    right.set_title(f"measured {run.pressure():.3e} vs predicted {target:.3e}")
    plt.tight_layout()
    plt.show()


widgets.interact(
    explore,
    n_particles=widgets.IntSlider(min=10, max=2000, step=10, value=200, description="N"),
    temperature=widgets.FloatSlider(min=50, max=1000, step=25, value=300.0, description="T (K)"),
    mass_amu=widgets.FloatSlider(min=1, max=200, step=1, value=39.948, description="m (amu)"),
    box_microns=widgets.FloatSlider(min=0.5, max=3.0, step=0.1, value=1.0, description="L (um)"),
);

## Check your understanding

Run the cell below for the auto-graded quiz. The same questions, with written explanations
for every option, are on the module page.

In [ ]:
import json
from pathlib import Path

quiz_path = Path("..") / "_quiz" / "04-pressure.json"
if quiz_path.exists():
    from jupyterquiz import display_quiz

    # Parsed here with an explicit encoding: jupyterquiz opens the file with the platform
    # default, which cannot decode the Hebrew edition of this notebook on Windows.
    display_quiz(json.loads(quiz_path.read_text(encoding="utf-8")))
else:
    print("Quiz not generated yet — run: uv run python scripts/render_quizzes.py")

## Before you leave

Write a few sentences on each, in the cell below.

1. What did you predict that turned out to be wrong, and what specifically was the flaw in
   your reasoning?
2. This model has no particle–particle collisions at all. Name one conclusion from today that
   is therefore **not** established by the simulation, even though the numbers agreed.
3. Explain, without equations, why relative fluctuations shrink with $N$ while absolute
   fluctuations grow.

**Your answers:**

1.
2.
3.